In [1]:
!pip install -q llama-index
!pip install -q llama-index-retrievers-bm25

In [4]:
import sys
import time
import pickle
import logging
from pathlib import Path
from typing import List, Optional, Tuple, Union
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from llama_index.core import Document
# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents, select_random_documents
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)
from contextlib import contextmanager

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

logging.basicConfig(level=logging.INFO)

class SparseBM25Vectorizer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(binary=False, lowercase=True, token_pattern=r'\b\w+\b')
        self.doc_len = None
        self.avgdl = None
        self.idf = None
        self.doc_freq = None

    def fit(self, documents: List[Union[Document, str]]):
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.fit_transform(texts)
        self.doc_len = X.sum(axis=1).A1
        self.avgdl = self.doc_len.mean()
        n_docs = X.shape[0]
        self.doc_freq = X.sum(axis=0).A1
        self.idf = np.log((n_docs - self.doc_freq + 0.5) / (self.doc_freq + 0.5) + 1.0)
        return self

    def transform(self, documents: Union[List[Union[Document, str]], str]):
        if isinstance(documents, str):
            documents = [documents]
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.transform(texts)
        return self._bm25_sparse(X)

    def _bm25_sparse(self, X):
        rows, cols = X.nonzero()
        data = X.data
        
        doc_len = self.doc_len[rows]
        idf = self.idf[cols]
        
        numerator = data * (self.k1 + 1)
        denominator = data + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
        
        scores = idf * numerator / denominator
        return csr_matrix((scores, (rows, cols)), shape=X.shape)

class SparseBM25Retriever:
    def __init__(self, documents, vectorizer, embeddings):
        self.documents = documents
        self.vectorizer = vectorizer
        self.embeddings = embeddings

    def retrieve(self, query: str, top_k=10):
        """
        Efficient retrieval of top-K relevant documents using sparse BM25.
        """
        query_embedding = self.vectorizer.transform([query])  # Ensure it's a sparse matrix
        
        # Compute sparse cosine similarity (dot product is enough in BM25 space)
        scores = query_embedding @ self.embeddings.T  # Sparse matrix multiplication
        scores = np.array(scores.toarray()).flatten()  # Convert to dense array
        
        # Get top-k indices efficiently using argpartition (O(N log K) instead of O(N log N))
        top_k_idx = np.argpartition(scores, -top_k)[-top_k:]  # Get top-K without sorting everything
        top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]  # Sort top-K results

        # Retrieve top results
        results = [
            {
                "item_key": self.documents[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": self.documents[idx].text,
                "score": scores[idx]
            }
            for idx in top_k_idx
        ]

        return results

def main(file_identifier, batch_size=500, top_k=10):
    start_total = time.time()

    # Define index paths for "texto" and "resumen"
    index_path_texto = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes/texto')
    index_path_resumen = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes/resumen')

    # Load persisted documents
    with open(index_path_texto / "documents.pkl", "rb") as f:
        documents_texto = pickle.load(f)
    with open(index_path_resumen / "documents.pkl", "rb") as f:
        documents_resumen = pickle.load(f)

    # Load persisted vectorizers and embeddings
    with open(index_path_texto / "vectorizer.pkl", "rb") as f:
        vectorizer_texto = pickle.load(f)
    embeddings_texto = load_npz(index_path_texto / "embeddings.npz")

    with open(index_path_resumen / "vectorizer.pkl", "rb") as f:
        vectorizer_resumen = pickle.load(f)
    # (We won’t use vectorizer_resumen here since we’re reusing vectorizer_texto for retrieval.)

    # Build a mapping from texto item keys to their indices for fast lookup
    texto_mapping = {doc.metadata.get("item_key"): idx for idx, doc in enumerate(documents_texto)}

    # Select a random sample from the resumen documents
    sample_size = min(len(documents_resumen), 16590)
    random_resumen = select_random_documents(documents_resumen, sample_size)
    resumen_keys, resumen_texts = zip(*random_resumen)  # unzip into separate lists

    all_results = []
    total_batches = (sample_size + batch_size - 1) // batch_size
    start_batches = time.time()

    # Batch processing: transform and compute scores in batches
    for batch_num, start_idx in enumerate(range(0, sample_size, batch_size), 1):
        end_idx = min(start_idx + batch_size, sample_size)
        batch_keys = resumen_keys[start_idx:end_idx]
        batch_texts = resumen_texts[start_idx:end_idx]

        # Batch-transform resumen texts using vectorizer_texto
        batch_query_embeddings = vectorizer_texto.transform(batch_texts)
        # Compute similarity scores in one sparse matrix multiplication
        # scores_batch: shape (batch_size, number of texto documents)
        scores_batch = (batch_query_embeddings @ embeddings_texto.T).toarray()

        # Process each query in the batch
        for i, score_row in enumerate(scores_batch):
            overall_idx = start_idx + i
            item_key = batch_keys[i]
            resumen_text = batch_texts[i]

            # Extract top_k indices and sort them by score descending
            topk_idx_unsorted = np.argpartition(score_row, -top_k)[-top_k:]
            topk_idx = topk_idx_unsorted[np.argsort(score_row[topk_idx_unsorted])[::-1]]

            # Build the list of top_k results
            texto_results = [{
                "item_key": documents_texto[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": documents_texto[idx].text,
                "score": score_row[idx]
            } for idx in topk_idx]

            # Get target score from matching texto document (if exists)
            texto_index = texto_mapping.get(item_key, None)
            if texto_index is not None:
                target_score = score_row[texto_index]
                target_text = documents_texto[texto_index].text
            else:
                target_score = 0.0
                target_text = "No matching texto found"

            # Append results for this resumen document
            all_results.append({
                'resumen_item_key': item_key,
                'resumen_text': resumen_text,
                'target_text': target_text,
                'target_score': target_score,
                'texto_results': texto_results
            })

        # Logging progress for each batch
        elapsed_batches = time.time() - start_batches
        batches_remaining = total_batches - batch_num
        avg_batch_time = elapsed_batches / batch_num
        estimated_remaining_sec = batches_remaining * avg_batch_time
        est_min, est_sec = divmod(estimated_remaining_sec, 60)
        logging.info(
            f"Batch {batch_num}/{total_batches} processed. "
            f"Estimated time to completion: {int(est_min)}m {int(est_sec)}s"
        )

    # Perform overall analysis
    analysis_results = perform_analysis(all_results)
    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")

    # Perform single-match analysis and print results
    metrics = perform_single_match_analysis(all_results, top_k=top_k)
    print_single_match_analysis(metrics, top_k=top_k)

    # Perform prefix-matching analysis
    with timer("Performing prefix-matching analysis"):
        prefix_length = 6  # Match first 6 characters of item_key
        prefix_match_metrics = perform_prefix_match_analysis(
            all_results, 
            top_k=10, 
            prefix_length=prefix_length
        )
        print_prefix_match_analysis(
            prefix_match_metrics, 
            top_k=10, 
            prefix_length=prefix_length
        )

    # Print some sample retrieval results
    print_sample_results(all_results)

    total_elapsed = time.time() - start_total
    tot_min, tot_sec = divmod(total_elapsed, 60)
    print(f"\nTotal execution time: {int(tot_min)}m {int(tot_sec)}s")


In [5]:
main('OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 17s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 14s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 13s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 13s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 12s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.2889
Precision: 0.0289
F1 Score: 0.0525
Mean Reciprocal Rank (MRR): 0.0934
Mean Average Precision (MAP): 0.0934

Single-Match Analysis Results:
Success@10             : 28.89%
MRR: 0.0934
Mean Position: 5.03
Median Position: 5.0
Perfect Matches: 3.41%

Position Distribution:
------------------------------
Position_1: 3.41%
Position_2: 3.45%
Position_3: 3.48%
Position_4: 3.63%
Position_5: 2.71%
Position_6: 2.99%
Position_7: 2.26%
Position_8: 2.42%
Position_9: 2.27%
Position_10: 2.29%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 71.82%
Prefix_MRR: 0.4799
Prefix_Mean_Position: 2.71
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 38.90%

Prefix Position Distribution:
----------------------------------------
Position 1    : 38.90%
Position 2    : 7.89%
Position 3    : 4.77%
Position 4    : 4.54%
Position 5    : 4.14%
Position 6    : 3.68%
Position 7    : 2.77%
Position 8    : 2.25%
Position 9    : 1.62%
Position 10 